In [ ]:
import os
import pandas as pd
import re, string, nltk
from datasets import load_dataset
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer
from nltk import word_tokenize, pos_tag
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

In [6]:
# Создание папки для nltk данных, если её нет
nltk_data_dir = os.path.expanduser('../nltk_data')
if not os.path.exists(nltk_data_dir):
    os.makedirs(nltk_data_dir)

# Добавление пути в nltk
nltk.data.path.append(nltk_data_dir)

# Загрузка всех необходимых ресурсов
resources = ['punkt', 'wordnet', 'omw-1.4', 'punkt_tab', 
             'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng', 'stopwords']

for resource in resources:
    try:
        nltk.download(resource, download_dir=nltk_data_dir, quiet=False)
    except:
        print(f"Ресурс {resource} уже загружен или произошла ошибка")

[nltk_data] Downloading package punkt to ../nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to ../nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to ../nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to ../nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     ../nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     ../nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to ../nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [10]:
def clean_text(text):

    # Приведение к нижнему регистру
    text = text.lower()

    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[{}[\]()<>]', '', text)
    text = re.sub(r'\d+\.\d+\.\d+\.\d+', '', text)  # IP-адреса
    # Удаление email-адресов
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    text = re.sub(email_pattern, '', text)
    # Удаление путей к файлам
    text = re.sub(r'\S+/\S+/\S+', '', text)
    text = re.sub(r'\S+\.\S+/\S+', '', text)
    # Удаление упоминаний (@username), хештеги
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    # ОЧИСТКА ОТ КАВЫЧЕК И СПЕЦСИМВОЛОВ
    text = re.sub(r'[`"\']{2,}', ' ', text)  # удвоенные кавычки
    text = re.sub(r'[`"\']', ' ', text)      # одиночные кавычки
    text = re.sub(r'[\[\]{}()<>]', ' ', text)  # скобки
    # Обработка технических терминов
    # Замена "scsi-1" на "scsi1" или оставляем как есть
    text = re.sub(r'(\w+)-(\d+)', r'\1\2', text)  # убираем дефис в терминах
    text = re.sub(r'(\d+)-(\w+)', r'\1\2', text)
    # Удаление текстовых смайликов
    smile_pattern = r'[:;=][\-^]?[)D\(\[\]pP]+'
    text = re.sub(smile_pattern, '', text)
    # Удаление повторяющихся знаков пунктуации (!!!, ???, ...)
    text = re.sub(r'([!?.,])\1+', r'\1', text)  # !!! -> !
    text = re.sub(r'-{2,}', ' ', text)
    # Удаление телефонных номеров (опционально)
    text = re.sub(r'\+?\d[\d\s\-\(\)]{7,}\d', '', text)
    text = re.sub(r'[^\w\s]', ' ', text) 
    # Удаление лишних пробелов
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    
    return text

from nltk.corpus import wordnet
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))
additional_stops = {
    'would', 'could', 'should', 'might', 'may', 'get', 'go', 'see', 
    'know', 'like', 'want', 'need', 'say', 'think', 'come', 'take',
    'use', 'make', 'well', 'also', 'even', 'many', 'much', 'still',
    'however', 'though', 'although', 'since', 'yet', 'already'
}
stop_words.update(additional_stops)

def get_wordnet_pos(tag):
    if tag.startswith('J'): return wordnet.ADJ
    elif tag.startswith('V'): return wordnet.VERB
    elif tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

def remove_stopwords(tokens):
    return [token for token in tokens if token not in stop_words]


# Применение лемматизации
lemmatizer = nltk.WordNetLemmatizer()
# Функция предобработки с лемматизацией
def preprocess_with_lemmatization(text, pos_filter="all"):
    

    # Токенизация
    tokens = word_tokenize(clean_text(text))
    tagged = pos_tag(tokens)

    lemmatized_tokens = []

    for token, tag in tagged:
        if token in stop_words or token in string.punctuation:
            continue
        
        is_noun_adj = tag.startswith('N') or tag.startswith('J')
        if pos_filter == "nouns_adj" and not is_noun_adj:
            continue

        lemmatized_tokens.append(lemmatizer.lemmatize(token, get_wordnet_pos(tag)))
    
    return " ".join(lemmatized_tokens)

stemmer = PorterStemmer()
def preprocess_with_stemming(text):
    # Используем твою функцию очистки
    tokens = word_tokenize(clean_text(text))
    res = []
    for token in tokens:
        if token in stop_words or token in string.punctuation: continue
        # Просто отрезаем окончания
        res.append(stemmer.stem(token))
    return " ".join(res)

In [11]:
dataset = load_dataset("SetFit/20_newsgroups")
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

Repo card metadata block was not found. Setting CardData to empty.


In [12]:
print("Лемматизация данных...")

train_df['text_stem'] = train_df['text'].apply(preprocess_with_stemming)
test_df['text_stem'] = test_df['text'].apply(preprocess_with_stemming)

train_df['text_simple'] = train_df['text'].apply(lambda x: preprocess_with_lemmatization(x, 'all'))
train_df['text_pos'] = train_df['text'].apply(lambda x: preprocess_with_lemmatization(x, 'nouns_adj'))

test_df['text_simple'] = test_df['text'].apply(lambda x: preprocess_with_lemmatization(x, 'all'))
test_df['text_pos'] = test_df['text'].apply(lambda x: preprocess_with_lemmatization(x, 'nouns_adj'))

Лемматизация данных...


In [13]:
def run_benchmarks(train_col, test_col):
    results = []
    vectorizers = {
        'Binary': CountVectorizer(binary=True),
        'Count': CountVectorizer(binary=False),
        'TF-IDF': TfidfVectorizer()
    }
    
    for v_name, vec in vectorizers.items():
        # Подготовка признаков
        X_train = vec.fit_transform(train_df[train_col])
        X_test = vec.transform(test_df[test_col])
        y_train, y_test = train_df['label'], test_df['label']
        
        models = {
            "Decision Tree": DecisionTreeClassifier(max_depth=20, random_state=42),
            "Random Forest": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
        }
        
        for name, clf in models.items():
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)
            
            # Собираем все три вида F1-score
            results.append({
                "Предобработка": train_col,
                "Вектор": v_name,
                "Модель": name,
                "F1 Macro": f1_score(y_test, y_pred, average='macro'),
                "F1 Micro": f1_score(y_test, y_pred, average='micro'),
                "F1 Weighted (Avg)": f1_score(y_test, y_pred, average='weighted')
            })
    return results

In [14]:
results_all = (
    run_benchmarks('text_simple', 'text_simple') + # Лемматизация
    run_benchmarks('text_pos', 'text_pos') +       # Лемма + Сущ/Прил
    run_benchmarks('text_stem', 'text_stem')       # Стемминг
)

df_results = pd.DataFrame(results_all).sort_values('F1 Weighted (Avg)', ascending=False)
display(df_results)

,Предобработка,Вектор,Модель,F1 Macro,F1 Micro,F1 Weighted (Avg)
17,text_stem,TF-IDF,Random Forest,0.604217,0.625199,0.617199
5,text_simple,TF-IDF,Random Forest,0.603180,0.621349,0.615623
15,text_stem,Count,Random Forest,0.598546,0.618163,0.611166
3,text_simple,Count,Random Forest,0.597543,0.614843,0.609180
1,text_simple,Binary,Random Forest,0.595973,0.613117,0.607935
13,text_stem,Binary,Random Forest,0.593744,0.615773,0.607463
11,text_pos,TF-IDF,Random Forest,0.591846,0.609533,0.604904
9,text_pos,Count,Random Forest,0.584161,0.601965,0.596633
7,text_pos,Binary,Random Forest,0.579557,0.598115,0.591877
12,text_stem,Binary,Decision Tree,0.342045,0.302974,0.353793
